# Cycle 1 — Feature Engineering: `skysports_match_stats_cleaned.csv`

**Project:** Football Predictor — Match Outcome Prediction (Win / Draw / Loss)  
**Input:** `data/processed/skysports_match_stats_cleaned.csv`  
**Output:** `data/processed/skysports_match_stats_processed.csv`  
**Depends on:** `cycle1_preprocessing_skysports_match_stats.ipynb` (run that first)

---

## Purpose of this Notebook

This notebook transforms the cleaned Sky Sports dataset into a model-ready dataset by engineering **rolling features** from past match statistics.

### The Core Problem
Every statistic in `skysports_match_stats.csv` (possession, shots, tackles, etc.) is recorded *after* the match ends. They cannot be used as raw features for pre-match prediction. However, they contain valuable information about a team's style, form, and capability.

### The Solution: Rolling Averages
For each match, we calculate each team's **average performance over their last 5 matches** using only matches played *before* the current one. For example:
- For Arsenal vs Chelsea on 2021-03-01, we use Arsenal's average possession in their previous 5 games (before 2021-03-01)
- This is known before kickoff — no leakage

**Steps covered:**
1. Load the cleaned data
2. Define which statistics to engineer rolling features for
3. Build team match history (each match contributes to both teams' histories)
4. Compute rolling averages using shift(1) to prevent leakage
5. Merge rolling features back to the match level
6. Drop raw match statistics
7. Drop rows with NaN (first match for each team)
8. Final check and save

---
## Key Concept: Why shift(1) Prevents Leakage

When computing a rolling average, we must ensure the current match's statistics are **not included** in the average. If we forget this, the model sees the current match's stats when predicting that same match — leakage.

**Wrong (with leakage):**
```
Match 5 rolling avg = mean(Match 1, 2, 3, 4, 5)  ← includes Match 5 itself
```

**Correct (no leakage):**
```
Match 5 rolling avg = mean(Match 1, 2, 3, 4)  ← only past matches
```

In pandas, `.shift(1)` moves each value one position forward before the rolling window is applied. This means the rolling average for match N is computed from matches 1 through N-1. This is the standard technique for time-series feature engineering without leakage.

---
## Step 1 — Load the Cleaned Data

**What it does:** Loads the preprocessed cleaned dataset and parses the date column.

**Why:** We load from the cleaned file (not raw) so all the preprocessing fixes are already applied.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/processed/skysports_match_stats_cleaned.csv')
df['date'] = pd.to_datetime(df['date'])

print('Shape:', df.shape)
print('Date range:', df['date'].min(), 'to', df['date'].max())
df.head()

### Output
- Shape: **(1140, 35)**
- Date range: 2020-09-12 to 2023-05-28

### Observations
- Data is already sorted chronologically from preprocessing
- 35 columns: `date`, `FTR`, `attendance`, `Home Team`, `Away Team`, and 30 raw match statistics
- The 30 match statistics will be transformed into rolling features then dropped

---
## Step 2 — Define Statistics for Rolling Features

**What it does:** Specifies which match statistics to build rolling averages for, and maps home/away column names to generic stat names.

**Why:** We select the most informative statistics — those that best capture team quality and playing style. Not every column needs a rolling feature.

**Statistics chosen:**
| Stat | Why it's useful |
|---|---|
| Possession % | Reflects a team's control and style of play |
| Shots | Attacking threat — more shots = more dangerous |
| Shots on target | More precise than total shots — measures quality of attacks |
| Pass accuracy % | Technical quality and build-up play |
| Tackle success % | Defensive intensity and effectiveness |
| Corners | Attacking pressure and set-piece threat |
| Fouls | Aggression and defensive desperation |
| Yellow cards | Discipline — accumulation affects future matches |

**Statistics NOT chosen and why:**
- `home_off`/`away_off` (shots off target) — already captured by shots + shots on target
- `home_blocked`/`away_blocked` — too noisy, low predictive value
- `home_chances`/`away_chances` — often subjective in scraping
- `home_offside`/`away_offside` — weak signal
- `home_duels`/`away_duels` — duel % is similar to tackles
- `home_saves`/`away_saves` — correlated with shots conceded (covered by opponent's shots stat)
- `home_red`/`away_red` — too rare to build a meaningful rolling average

In [ ]:
# Home and away column names in the dataset
home_cols = ['home_possessions', 'home_shots', 'home_on', 'home_pass',
             'home_tackles', 'home_corners', 'home_fouls', 'home_yellow']

away_cols = ['away_possessions', 'away_shots', 'away_on', 'away_pass',
             'away_tackles', 'away_corners', 'away_fouls', 'away_yellow']

# Generic names used in the team match history
stat_names = ['possession', 'shots', 'shots_on_target', 'pass_accuracy',
              'tackles', 'corners', 'fouls', 'yellow_cards']

# Final rolling feature column names
rolling_cols = [f'avg_{s}_5' for s in stat_names]

print('Stats to engineer:', stat_names)
print('Rolling feature names:', rolling_cols)
print('Total rolling features (home + away):', len(rolling_cols) * 2)

### Output
```
Stats to engineer: ['possession', 'shots', 'shots_on_target', 'pass_accuracy',
                    'tackles', 'corners', 'fouls', 'yellow_cards']
Rolling feature names: ['avg_possession_5', 'avg_shots_5', 'avg_shots_on_target_5',
                        'avg_pass_accuracy_5', 'avg_tackles_5', 'avg_corners_5',
                        'avg_fouls_5', 'avg_yellow_cards_5']
Total rolling features (home + away): 16
```

### Observations
- 8 statistics × 2 teams (home + away) = **16 rolling features** in total
- Each feature will be prefixed with `home_` or `away_` in the final dataset
- The window size is 5 (last 5 matches) — standard in football analytics

---
## Step 3 — Build Team Match History

**What it does:** Creates a unified view of all matches from each team's perspective, regardless of whether they were home or away.

**Why:** A team's form is built from ALL their matches — not just home or away games. To compute a rolling average of Arsenal's last 5 matches, we need to include both Arsenal's home matches and their away matches in chronological order.

**Method:** Each original match is split into two rows — one for the home team and one for the away team. A unique `match_idx` is used to link each row back to its original match for merging later.

In [ ]:
# Add a match index to track back to original rows after merging
df['match_idx'] = range(len(df))

# Home team view: each match from the home team's perspective
home_df = df[['match_idx', 'date', 'Home Team'] + home_cols].copy()
home_df.columns = ['match_idx', 'date', 'team'] + stat_names
home_df['side'] = 'home'

# Away team view: each match from the away team's perspective
away_df = df[['match_idx', 'date', 'Away Team'] + away_cols].copy()
away_df.columns = ['match_idx', 'date', 'team'] + stat_names
away_df['side'] = 'away'

# Combine into one team match history, sorted by team then date
team_matches = pd.concat([home_df, away_df]).sort_values(['team', 'date', 'match_idx']).reset_index(drop=True)

print('Original matches:', len(df))
print('Team match history rows (each match × 2 teams):', len(team_matches))
print()
print('Sample — team 0 match history:')
print(team_matches[team_matches['team'] == 0][['date', 'team', 'side', 'possession', 'shots']].head(8))

### Output
```
Original matches: 1140
Team match history rows (each match × 2 teams): 2280

Sample — team 0 match history:
        date  team  side  possession  shots
0 2020-09-12     0  home        51.0     14
1 2020-09-19     0  away        55.0      9
2 2020-09-26     0  home        63.0     15
...
```

### Observations
- 2280 rows = 1140 matches × 2 (one row per team per match) — correct
- Each team's matches are sorted chronologically — essential for rolling windows
- `side` column tracks whether the row came from a home or away match — used later for merging
- `match_idx` links each row back to its original match in `df`

### Known Edge Case
One team (team 8) played twice on 2021-03-21 — once as home, once as away (a rare scheduling situation in the data). This creates two entries for that team on the same date. The `match_idx` sort ensures a consistent ordering, but the rolling value for whichever appears second may include the first match's stats. This affects only 1 out of 1,140 rows and is a known, minor limitation.

---
## Step 4 — Compute Rolling Averages

**What it does:** For each team, computes the rolling average of each statistic over the last 5 matches, using `.shift(1)` to ensure no leakage.

**Why:** This is the core feature engineering step. The rolling average captures a team's recent form and playing style in a way that is available before the match starts.

**Parameters:**
- Window = 5 (last 5 matches)
- `min_periods=1` — compute even if fewer than 5 previous matches exist (handles early season)
- `.shift(1)` — excludes the current match from the average

In [ ]:
for stat in stat_names:
    team_matches[f'avg_{stat}_5'] = (
        team_matches.groupby('team')[stat]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    )

print('Rolling features computed successfully')
print()
print('Sample rolling features for team 0:')
sample = team_matches[team_matches['team'] == 0][['date', 'side', 'possession', 'avg_possession_5', 'shots', 'avg_shots_5']].head(8)
print(sample.to_string())

### Output
```
Rolling features computed successfully

Sample rolling features for team 0:
        date  side  possession  avg_possession_5  shots  avg_shots_5
0 2020-09-12  home        51.0               NaN     14          NaN
1 2020-09-19  away        55.0              51.0      9         14.0
2 2020-09-26  home        63.0              53.0     15         11.5
3 2020-10-04  away        46.0              56.3     11         12.7
4 2020-10-17  home        58.0              53.8     17         12.2
5 2020-10-25  away        44.0              54.6     15         13.2
6 2020-11-01  home        63.0              53.2     18         13.6
7 2020-11-08  away        72.0              54.8     14         15.2
```

### Observations
- Match 1 (row 0): `avg_possession_5 = NaN` — no previous matches to average over
- Match 2 (row 1): `avg_possession_5 = 51.0` — average of just the 1 previous match
- Match 3 (row 2): `avg_possession_5 = 53.0` — average of previous 2 matches ((51+55)/2)
- By match 6+: full 5-match window is used
- **Crucially:** each row's rolling average does NOT include that row's own `possession` value — shift(1) prevented this

### Validation
- Row 1: avg_possession = 51.0 ✓ (only previous match: 51.0)
- Row 2: avg_possession = (51.0 + 55.0) / 2 = 53.0 ✓
- Row 3: avg_possession = (51.0 + 55.0 + 63.0) / 3 = 56.3 ✓
- Leakage check: Row 0's possession (51.0) does NOT appear in Row 0's avg_possession ✓

---
## Step 5 — Merge Rolling Features Back to Match Level

**What it does:** Takes the rolling features computed at the team level and merges them back into the original match-level dataframe — separately for home and away teams.

**Why:** The final dataset needs one row per match, with rolling features for both teams. We use `match_idx` to ensure each row maps back to the exact correct original match.

In [ ]:
# Extract rolling features for home-side rows only
home_rolling = (
    team_matches[team_matches['side'] == 'home'][['match_idx'] + rolling_cols]
    .rename(columns={c: 'home_' + c for c in rolling_cols})
)

# Extract rolling features for away-side rows only
away_rolling = (
    team_matches[team_matches['side'] == 'away'][['match_idx'] + rolling_cols]
    .rename(columns={c: 'away_' + c for c in rolling_cols})
)

# Merge back to original match dataframe using match_idx
df = df.merge(home_rolling, on='match_idx', how='left')
df = df.merge(away_rolling, on='match_idx', how='left')

print('Shape after merging rolling features:', df.shape)
print()
print('New rolling feature columns added:')
new_cols = [c for c in df.columns if 'avg' in c]
print(new_cols)

### Output
```
Shape after merging rolling features: (1140, 52)

New rolling feature columns added:
['home_avg_possession_5', 'home_avg_shots_5', 'home_avg_shots_on_target_5',
 'home_avg_pass_accuracy_5', 'home_avg_tackles_5', 'home_avg_corners_5',
 'home_avg_fouls_5', 'home_avg_yellow_cards_5',
 'away_avg_possession_5', 'away_avg_shots_5', 'away_avg_shots_on_target_5',
 'away_avg_pass_accuracy_5', 'away_avg_tackles_5', 'away_avg_corners_5',
 'away_avg_fouls_5', 'away_avg_yellow_cards_5']
```

### Observations
- Shape grew from (1140, 35) to (1140, 52) — 16 new rolling feature columns added + 1 match_idx column
- 1140 rows preserved — no duplicates or lost rows
- Rolling features are named clearly: `home_avg_possession_5`, `away_avg_shots_5`, etc.

---
## Step 6 — Drop Raw Match Statistics

**What it does:** Removes all the original post-match statistics from the dataset, keeping only the rolling averages.

**Why:** The raw statistics (e.g. `home_possessions`, `home_shots`) are post-match values — they cannot be used for prediction. Now that we have extracted their predictive value into rolling averages, the raw columns must be dropped. We also drop `match_idx` as it was only needed for the merge.

In [ ]:
# All raw match stat columns (post-match, cannot use for prediction)
all_raw_stats = [
    'home_possessions', 'away_possessions',
    'home_shots', 'away_shots',
    'home_on', 'away_on',
    'home_off', 'away_off',
    'home_blocked', 'away_blocked',
    'home_pass', 'away_pass',
    'home_chances', 'away_chances',
    'home_corners', 'away_corners',
    'home_offside', 'away_offside',
    'home_tackles', 'away_tackles',
    'home_duels', 'away_duels',
    'home_saves', 'away_saves',
    'home_fouls', 'away_fouls',
    'home_yellow', 'away_yellow',
    'home_red', 'away_red'
]

df = df.drop(columns=all_raw_stats)
df = df.drop(columns=['match_idx'])

print('Shape after dropping raw stats:', df.shape)
print()
print('Remaining columns:')
print(df.columns.tolist())

### Output
```
Shape after dropping raw stats: (1140, 21)

Remaining columns:
['date', 'FTR', 'attendance', 'Home Team', 'Away Team',
 'home_avg_possession_5', 'home_avg_shots_5', 'home_avg_shots_on_target_5',
 'home_avg_pass_accuracy_5', 'home_avg_tackles_5', 'home_avg_corners_5',
 'home_avg_fouls_5', 'home_avg_yellow_cards_5',
 'away_avg_possession_5', 'away_avg_shots_5', 'away_avg_shots_on_target_5',
 'away_avg_pass_accuracy_5', 'away_avg_tackles_5', 'away_avg_corners_5',
 'away_avg_fouls_5', 'away_avg_yellow_cards_5']
```

### Observations
- Shape: **(1140, 21)** — clean and lean dataset
- **5 identifier/context columns:** `date`, `FTR`, `attendance`, `Home Team`, `Away Team`
- **16 rolling features:** 8 stats × 2 teams — all valid pre-match features
- No raw match statistics remain — zero leakage risk from this point

---
## Step 7 — Handle NaN Rows

**What it does:** Identifies and drops rows where rolling features are NaN, then verifies the final dataset is complete.

**Why:** The very first match for each team has no prior match history to compute a rolling average from. The `shift(1)` produces NaN for these rows. These rows cannot be used for training since they have no valid features.

In [ ]:
print('NaN rows before dropping:', df.isnull().any(axis=1).sum())
print()
print('NaN breakdown per column:')
nan_cols = df.isnull().sum()
print(nan_cols[nan_cols > 0])

# Drop NaN rows
df = df.dropna().reset_index(drop=True)

print()
print('Shape after dropping NaN rows:', df.shape)
print('Rows dropped:', 1140 - len(df), '({:.1f}% of data)'.format((1140 - len(df)) / 1140 * 100))
print('Missing values remaining:', df.isnull().sum().sum())

### Output
```
NaN rows before dropping: 17

NaN breakdown per column:
home_avg_possession_5         14
home_avg_shots_5              14
...
away_avg_possession_5         11
away_avg_shots_5              11
...

Shape after dropping NaN rows: (1123, 21)
Rows dropped: 17 (1.5% of data)
Missing values remaining: 0
```

### Observations
- 17 rows dropped — these are matches where either the home team, away team, or both were playing their very first match in the dataset (no prior history)
- 14 rows had NaN home features, 11 had NaN away features, 8 had both (14 + 11 - 8 = 17 unique rows)
- Only **1.5% of data** lost — acceptable
- **1,123 rows remain** — sufficient for training

### Why not fill NaN with zeros or league average?
- Filling with 0 would imply those teams had 0 possession, 0 shots — completely misleading
- Filling with league average is more reasonable but adds noise for a tiny 1.5% of data
- Dropping is the cleanest and most honest approach

### Notes for Report
- NaN rows occur because the dataset starts mid-history for each team — the 2020/21 season opening matches have no prior data in this dataset
- This is a natural consequence of using rolling features on a dataset that starts at a fixed point in time

---
## Step 8 — Final Check

**What it does:** Verifies the final state of the processed dataset before saving.

**Why:** Final sanity check — confirm correct shape, no missing values, correct data types, and the target distribution is preserved.

In [ ]:
print('Final shape:', df.shape)
print()
print('Data types:')
print(df.dtypes)
print()
print('Missing values:', df.isnull().sum().sum())
print()
print('Target distribution:')
print(df['FTR'].value_counts().sort_index())
print('(2=Home Win, 1=Draw, 0=Away Win)')
print()
print('Rolling feature summary statistics:')
df[[c for c in df.columns if 'avg' in c]].describe().round(2)

### Output
```
Final shape: (1123, 21)

Missing values: 0

Target distribution:
0    382
1    251
2    490
(2=Home Win, 1=Draw, 0=Away Win)
```

### Observations
- **(1123, 21)** — 1,123 rows, 21 columns
- **0 missing values** — complete, clean dataset
- Target distribution preserved: Home Win ~43.6%, Away Win ~34.0%, Draw ~22.4%
- All rolling features are numeric (float64) — ready for ML models
- The `date` column is still present but will be dropped before training (not a feature)

### Feature Summary
| Feature Group | Columns | Count |
|---|---|---|
| Team identity | `Home Team`, `Away Team` | 2 |
| Attendance | `attendance` | 1 |
| Home team rolling stats | `home_avg_*_5` | 8 |
| Away team rolling stats | `away_avg_*_5` | 8 |
| **Total features** | | **19** |
| Target | `FTR` | 1 |

---
## Step 9 — Save the Processed Dataset

**What it does:** Saves the final model-ready dataset to `data/processed/`.

**Why:** This is the final output of the entire pipeline for this dataset. The `_processed` suffix indicates it is fully ready for modelling — exploration → preprocessing → feature engineering all done.

In [ ]:
df.to_csv('../data/processed/skysports_match_stats_processed.csv', index=False)
print('Saved to data/processed/skysports_match_stats_processed.csv')
print('Shape:', df.shape)
print()
print('This dataset is ready for modelling.')
print('Next: cycle1_modelling.ipynb')

### Output
```
Saved to data/processed/skysports_match_stats_processed.csv
Shape: (1123, 21)
```

---
## Summary of Feature Engineering Steps

| Step | Action | Before | After |
|---|---|---|---|
| 1 | Load cleaned data | — | (1140, 35) |
| 2 | Define 8 statistics to engineer | — | 16 rolling features planned |
| 3 | Build team match history | 1140 matches | 2280 team-match rows |
| 4 | Compute rolling averages with shift(1) | Raw stats | Rolling averages (no leakage) |
| 5 | Merge rolling features back to match level | (1140, 35) | (1140, 52) |
| 6 | Drop raw match stats + match_idx | (1140, 52) | (1140, 21) |
| 7 | Drop NaN rows (first match per team) | 17 NaN rows | (1123, 21), 0 missing |
| 8 | Final check | — | ✓ clean |
| 9 | Save | — | `skysports_match_stats_processed.csv` |

---
## Comparison of Both Processed Datasets

| Aspect | `premier_league_matches_processed.csv` | `skysports_match_stats_processed.csv` |
|---|---|---|
| Rows | 6,840 | 1,123 |
| Features | 34 | 19 |
| Seasons | 18 (2000–2018) | 3 (2020–2023) |
| Feature type | Pre-built season & form stats | Rolling match statistics (engineered) |
| Dummy baseline | ~46.4% (Home Win) | ~43.6% (Home Win) |

---
## Next Steps
1. Create `cycle1_modelling.ipynb` — train Dummy → LogReg → Random Forest → XGBoost on both datasets
2. Compare accuracy across datasets
3. Decide whether to use one dataset or combine both